In [3]:
import pandas as pd
import requests
from datetime import datetime , timedelta
import os
from mecoda_minka import get_obs, get_dfs
import folium
from folium.plugins import HeatMap
from html2image import Html2Image


API_PATH = "https://api.minka-sdg.org/v1"

os.makedirs("data", exist_ok=True)


# 1. Construir un dataframe de métricas principales del proyecto y variación en el último mes.

El dataframe resultante tendrá esta forma:

```python
metric, number_today, number_in_last_month
observations, 1521, 23
observers, 124, 2
identifiers, 26, 0
species, 462, 5
```

* number_today = dato actual de observaciones, observadores, identificadores, especies.
* number_in_last_month = dato de fecha actual menos dato registrado 30 días antes (variación en los últimos 30 días).

Usamos llamadas a la API, porque son datos totales. 

Creamos un directorio "data" donde guardaremos todos los csv que vamos a generar. Este dataframe lo guardamos como "data/main_metrics.cvs", sin incluir los índices (index=False).


In [4]:
def get_main_metrics(id_project):

    """
    Obtiene las métricas principales de un proyecto y el aumento en el mes anterior.
    :param id_project: ID del proyecto
    :return: DataFrame con las métricas principales
    """

    session = requests.Session()

    # Primero he definido las fechas que se utilizaran para obtener las metricas : Defino el mes actual y el mes anterior

    # MES ACTUAL
    today = datetime.today()
   
    # MES ANTERIOR
    last_month_date = today - timedelta(days=30)
  
    # METRICAS PARA EL MES ACTUAL 

    url_observations_today = f'{API_PATH}/observations?project_id={id_project}'
    observations_results_today = session.get(url_observations_today).json()['total_results']

    url_observers_today = f'{API_PATH}/observations/observers?project_id={id_project}'
    observers_results_today = session.get(url_observers_today).json()['total_results']

    url_identifiers_today = f'{API_PATH}/observations/identifiers?project_id={id_project}'
    identifiers_results_today = session.get(url_identifiers_today).json()['total_results']

    url_species_today = f'{API_PATH}/observations/species_counts?project_id={id_project}'
    species_results_today = session.get(url_species_today).json()['total_results']

    # METRICAS PARA EL MES ANTERIOR

    url_observations_last = url_observations_today + f'&d1={last_month_date}'
    observations_results_last = session.get(url_observations_last).json()['total_results']

    url_observers_last = url_observers_today + f'&d1={last_month_date}'
    observers_results_last = session.get(url_observers_last).json()['total_results']

    url_identifiers_last = url_identifiers_today + f'&d1={last_month_date}'
    identifiers_results_last = session.get(url_identifiers_last).json()['total_results']

    url_species_last = url_species_today + f'&d1={last_month_date}'
    species_results_last = session.get(url_species_last).json()['total_results']

    # Aqui genero el dataframe para almacenar los datos (TODOS LOS DATOS SON CON EL RESEARCH GRADE)

    df_main_metrics = pd.DataFrame({
        'metric': ['observations', 'observers', 'identifiers', 'species'],
        'number_today': [observations_results_today, observers_results_today, identifiers_results_today, species_results_today],
        'number_in_last_month': [observations_results_last, observers_results_last, identifiers_results_last, species_results_last]
    })

    return df_main_metrics

In [5]:
get_main_metrics(264)

,metric,number_today,number_in_last_month
0,observations,5835,186
1,observers,161,18
2,identifiers,68,7
3,species,671,100


# 2. Evolución de las métricas principales

Construir un dataframe con esta forma:

```python
month, observations, observers, identifiers, species
2024-01, 185, 15, 7, 62
2024-02, 128, 3, 1, 32
...
```

Los datos no son acumulativos, son del mes en concreto. Lo sacaremos usando llamadas a la API.

Para ello puedes utilizar estas funciones, que te ayudarán a construirlo:

Esto nos devuelve un diccionario con los meses como clave y el último día del mes como valor. Así podemos usarlo con la función anterior:

Ahora hay que unir las dos funciones para sacar cada mes y de cada mes sacar los valores de las métricas. Eso nos da los resultados de un mes, que podemos guardar en un diccionario. Y luego unimos los diccionarios de cada mes en una lista de todos los meses. Te pongo debajo un ejemplo de uso con un mes.

Ahora hay que crear una función que itere por todos los elementos de la lista meses desde el inicio de MINKA y saque los datos para cada mes usando get_totals a un diccionario, los acumule en la lista y la lista la convierta a un dataframe.

Es decir, para cada elemento de los meses, usamos get_totals para sacar las métricas y las almacenamos. Si lo ves complicado lo hacemos juntos.

Ese dataframe lo guardamos como "data/monthly_metrics.csv".

In [6]:
# En esta casilla se genera la funcion que recorre los meses y obtiene las metricas de cada uno de ellos
def get_totals(project_id, year, month, kind="project", session=None):
    if session is None:
        session = requests.Session()

    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"

    total_obs = session.get(url_obs).json().get("total_results", 0)
    total_part = session.get(url_part).json().get("total_results", 0)
    total_ident = session.get(url_ident).json().get("total_results", 0)
    total_spe = session.get(url_spe).json().get("total_results", 0)

    return total_obs, total_part, total_ident, total_spe


def get_month_list(years: list) -> list:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
    return meses

def build_monthly_metrics(project_id):
    meses = get_month_list(range(2022, datetime.now().year + 1))
    total_metrics = []

    session = requests.Session()

    for mes in meses:
        year = int(mes.split("-")[0])
        month = int(mes.split("-")[1])

        print(f"Procesando {mes}...")

        try:
            total_obs, total_part, total_ident, total_spe = get_totals(
                project_id=project_id, year=year, month=month, kind="project", session=session
            )
        except Exception as e:
            print(f"Error procesando {mes}: {e}")
            continue


        # Generamos el diccionario para posteriormente añadirlos a la lista total_metrics 
        total_for_month = {
            "month": mes,
            "observations": total_obs,
            "observers": total_part,
            "identifiers": total_ident,
            "species": total_spe
        }

        total_metrics.append(total_for_month)

    df_monthly = pd.DataFrame(total_metrics)
    # Encontramos el primer índice donde 'observations' > 0 (inicio del proyecto)
    start_index = df_monthly[df_monthly['observations'] > 0].index[0]
    df_final = df_monthly.loc[start_index:].reset_index(drop=True)
    return df_final

In [7]:
df_monthly = build_monthly_metrics(264)

Procesando 2022-01...
Procesando 2022-02...
Procesando 2022-03...
Procesando 2022-04...
Procesando 2022-05...
Procesando 2022-06...
Procesando 2022-07...
Procesando 2022-08...
Procesando 2022-09...
Procesando 2022-10...
Procesando 2022-11...
Procesando 2022-12...
Procesando 2023-01...
Procesando 2023-02...
Procesando 2023-03...
Procesando 2023-04...
Procesando 2023-05...
Procesando 2023-06...
Procesando 2023-07...
Procesando 2023-08...
Procesando 2023-09...
Procesando 2023-10...
Procesando 2023-11...
Procesando 2023-12...
Procesando 2024-01...
Procesando 2024-02...
Procesando 2024-03...
Procesando 2024-04...
Procesando 2024-05...
Procesando 2024-06...
Procesando 2024-07...
Procesando 2024-08...
Procesando 2024-09...
Procesando 2024-10...
Procesando 2024-11...
Procesando 2024-12...
Procesando 2025-01...
Procesando 2025-02...
Procesando 2025-03...
Procesando 2025-04...
Procesando 2025-05...
Procesando 2025-06...


In [8]:
df_monthly

,month,observations,observers,identifiers,species
0,2022-06,96,4,9,41
1,2022-07,32,5,6,12
2,2022-08,146,6,7,46
3,2022-09,146,8,8,55
4,2022-10,87,3,12,58
5,2022-11,115,2,10,74
6,2022-12,47,3,7,38
7,2023-01,57,4,7,37
8,2023-02,46,2,8,34
9,2023-03,63,5,9,48


In [9]:
observations = get_obs(id_project=264, grade="research")
df_obs, df_photos = get_dfs(observations)

# df_obs.to_csv("data/observations.csv", index=False)
# df_photos.to_csv("data/photos.csv", index=False)

Generating list of observations:
https://api.minka-sdg.org/v1/observations?project_id=264&quality_grade=research&per_page=200
Total observations to download: 4700
Number of elements: 200
Number of elements: 400
Number of elements: 600
Number of elements: 800
Number of elements: 1000
Number of elements: 1200
Number of elements: 1400
Number of elements: 1600
Number of elements: 1800
Number of elements: 2000
Number of elements: 2200
Number of elements: 2400
Number of elements: 2600
Number of elements: 2800
Number of elements: 3000
Number of elements: 3200
Number of elements: 3400
Number of elements: 3600
Number of elements: 3800
Number of elements: 4000
Number of elements: 4200
Number of elements: 4400
Number of elements: 4600
Number of elements: 4700


# 3. Taxonomías

Descargamos todas las observaciones del proyecto, usando mecoda_minka. Generamos los dataframes de observaciones y de fotos. A partir de df_obs creamos una función que nos permita ver el número de observaciones por reino, filo, clase... Este rango se tiene que poder indicar como parámetro, para usar la misma función para cualquier rango.

Guardamos df_obs y df_photos como csv en la carpeta `data`. Y no guardamos los índices, como en los casos anteriores.

Creamos la función. Ten en cuenta las columnas de los rangos que tiene el dataframe de df_obs. En las columnas "kingdom", "phylum", "class",... están los rangos taxonómicos superiores a la identificación. El "taxon_name" es el identificado en la observación, y el "taxon_rank" el rango de la identificación. En las otras columnas están los rangos superiores. Esto lo hacemos juntos, que es un poco complicado de explicar por escrito.

In [10]:
def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    if rank_level not in df_obs.columns:
        if rank_level == "species":
            df_taxon_counts = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()
        else:
            raise ValueError(f"'{rank_level}' esta columna no existe en el DataFrame.")

    if rank_level in df_obs.columns:
        df_taxon_counts = df_obs[rank_level].value_counts().reset_index()
        df_taxon_counts.columns = ["taxon_name", "count"]

    # Si hay observaciones identificadas como este rango
    if rank_level != "species":
        df2 = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()

        if len(df2) > 0:

            # Concatenamos ambos DataFrames
            df_combined = pd.concat([df_taxon_counts, df2])

            # Agrupamos por taxon_name y sumamos los counts
            df_summed = df_combined.groupby('taxon_name', as_index=False)['count'].sum()
            
            df_sorted = df_summed.sort_values(by='count', ascending=False).reset_index(drop=True)
            
        else:
            df_sorted = df_taxon_counts
    else:
        df_sorted = df_taxon_counts

    df_sorted['taxon_rank'] = rank_level

    return df_sorted[['taxon_rank', 'taxon_name', 'count']]

In [11]:
get_taxon_count(df_obs, "kingdom")

,taxon_rank,taxon_name,count
0,kingdom,Animalia,3178
1,kingdom,Plantae,1400
2,kingdom,Chromista,115
3,kingdom,Fungi,7


In [12]:
get_taxon_count(df_obs, "phylum")

,taxon_rank,taxon_name,count
0,phylum,Tracheophyta,1283
1,phylum,Arthropoda,886
2,phylum,Mollusca,874
3,phylum,Chordata,872
4,phylum,Cnidaria,327
5,phylum,Echinodermata,158
6,phylum,Ochrophyta,112
7,phylum,Rhodophyta,97
8,phylum,Bryozoa,38
9,phylum,Chlorophyta,20


In [13]:
get_taxon_count(df_obs, "class")

,taxon_rank,taxon_name,count
0,class,Magnoliopsida,919
1,class,Aves,755
2,class,Insecta,701
3,class,Bivalvia,521
4,class,Liliopsida,354
5,class,Gastropoda,239
6,class,Echinoidea,140
7,class,Hydrozoa,126
8,class,Scyphozoa,120
9,class,Cephalopoda,114


In [14]:
get_taxon_count(df_obs, 'species')

,taxon_rank,taxon_name,count
0,species,Acanthocardia tuberculata,112
1,species,Spisula subtruncata,110
2,species,Velella velella,99
3,species,Stramonita haemastoma,91
4,species,Paracentrotus lividus,82
...,...,...,...
571,species,Echium plantagineum,1
572,species,Tropinota squalida,1
573,species,Phallusia fumigata,1
574,species,Celastrina argiolus,1


# 4. Especies vistas por primera vez en el proyecto desde el último informe (últimos 30 días)

A partir del df_obs podemos sacar este dato fácilmente. Toma el dataframe, ordénalo por fecha de observación, en orden ascendente (las primeras observaciones estarán más arriba). Ahora quédate solo con las primeras observaciones de cada especie. Es decir:
* Seleccionamos aquellas observaciones que hayan llegado al nivel de especie (columna "taxon_rank" == "species").
* Nos quedamos con la primera observación de cada especie, usando drop_duplicates()
```python
df_first = df_obs.drop_duplicates(subset=["taxon_name"], keep="first")
```
* Así nos quedaremos con la primera observación de cada especie. Ahora filtramos de esta tabla las que tengan fecha de observación mayor a hoy menos 30 días (vistas en los últimos 30 días).

Esas serán las especies nuevas observadas en los últimos 30 días.

Primero haz el proceso y luego lo conviertes a una función. Es decir, carga el df_obs y haz los pasos con él, cuando te haya salido ya lo conviertes en función.

In [15]:
def get_new_species(df_obs, last_days=30):
    """
    Obtiene las nuevas especies observadas en los últimos días.
    :param df_obs: DataFrame con las observaciones
    :param last_days: Número de días para considerar una especie como nueva
    :return: DataFrame con las nuevas especies
    """
    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")

    df_species = df_obs[df_obs["taxon_rank"] == "species"]

    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    
    df_new_species = df_first[df_first["observed_on"] > cutoff_date]

    return df_new_species

In [16]:
new_spe = get_new_species(df_obs)
new_spe[["taxon_name", "observed_on", "user_login"]]

,taxon_name,observed_on,user_login
97,Streptopelia turtur,2025-05-13,mediambient_ajelprat
121,Matthiola incana,2025-05-13,amb_platges
105,Petrosedum sediforme,2025-05-15,amb_platges
39,Eatoniana plumipes,2025-05-26,xasalva
38,Elaeagnus angustifolia,2025-05-26,xasalva
25,Otala punctata,2025-05-26,xasalva
24,Beosus maritimus,2025-05-26,xasalva
7,Euchloe crameri,2025-05-26,amb_platges
21,Euodynerus variegatus,2025-05-26,xasalva
19,Plutella xylostella,2025-05-26,xasalva


Función para sacar una foto de las nuevas especies

In [17]:
def get_photos_new_species(df_new_species, df_photos):
    # El dataframe df_new_species tiene las especies nuevas, con el id de cada observación
    # Filtramos el dataframe de fotos para quedarnos solo con las fotos de las especies nuevas, las de los ids de esas observaciones.
    # Puedes utilizar el método isin() de pandas para filtrar el dataframe df_photos
    # df_photos['id'].isin(df_new_species['id'])
    # Investiga el método isin() y cómo se utiliza para filtrar un dataframe
    # Nos quedaríamos solo con una foto para cada especie nueva, así que podemos usar el método drop_duplicates() de pandas con subset(['id'])
    
    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    return df_photos_new_species

In [18]:
df_new_species = get_new_species(df_obs, last_days=30)
df_new_species

,id,created_at,updated_at,observed_on,observed_on_time,iconic_taxon,taxon_id,taxon_rank,taxon_name,latitude,...,identifiers,num_identification_agreements,num_identification_disagreements,device,kingdom,phylum,class,order,family,genus
97,455105,2025-05-16,2025-05-16,2025-05-13,08:17:00,aves,251769,species,Streptopelia turtur,41.278583,...,"mediambient_ajelprat, xasalva",1,0,web,Animalia,Chordata,Aves,Columbiformes,Columbidae,Streptopelia
121,454457,2025-05-14,2025-05-14,2025-05-13,10:17:00,plantae,246073,species,Matthiola incana,41.417919,...,"amb_platges, amb_platges, xasalva",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Capparales,Brassicaceae,Matthiola
105,454767,2025-05-15,2025-05-15,2025-05-15,10:21:00,plantae,249268,species,Petrosedum sediforme,41.269528,...,"amb_platges, xasalva",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Rosales,Crassulaceae,Petrosedum
39,460145,2025-05-26,2025-05-27,2025-05-26,08:13:00,arachnida,264884,species,Eatoniana plumipes,41.264961,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Arachnida,Prostigmata,Erythraeidae,None
38,460147,2025-05-26,2025-05-27,2025-05-26,08:14:00,plantae,245585,species,Elaeagnus angustifolia,41.264961,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Rhamnales,Elaeagnaceae,Elaeagnus
25,460169,2025-05-26,2025-05-27,2025-05-26,08:30:00,mollusca,252590,species,Otala punctata,41.265011,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Mollusca,Gastropoda,Stylommatophora,Helicidae,Otala
24,460170,2025-05-26,2025-05-27,2025-05-26,08:30:00,insecta,105506,species,Beosus maritimus,41.265011,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Insecta,Hemiptera,Rhyparochromidae,Beosus
7,460659,2025-05-27,2025-05-27,2025-05-26,13:23:00,insecta,247375,species,Euchloe crameri,41.418238,...,"amb_platges, xasalva",1,0,web,Animalia,Arthropoda,Insecta,Lepidoptera,Pieridae,Euchloe
21,460174,2025-05-26,2025-05-27,2025-05-26,08:34:00,insecta,247590,species,Euodynerus variegatus,41.265011,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Insecta,Hymenoptera,Eumenidae,Euodynerus
19,460183,2025-05-26,2025-05-30,2025-05-26,08:40:00,insecta,242685,species,Plutella xylostella,41.265005,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Insecta,Lepidoptera,Plutellidae,Plutella


In [19]:
def download_photos(
    df_photos: pd.DataFrame, directorio: str = "minka_photos"
):
    """
    Function to download the photos resulting from the query.
    """
    # Create the folder, if it exists overwrite it
    if not os.path.exists(directorio):
        os.makedirs(directorio)

    session = requests.Session()

    # Iterate through the df_photos query result and download the photos in medium size
    for i, row in df_photos.iterrows():
        response = session.get(row["photos_medium_url"], stream=True)
        if response.status_code == 200:
            with open(f"{directorio}/{row['path']}", "wb") as out_file:
                out_file.write(response.content)
        del response

    # Even using .loc, we get a SettingWithCopyWarning message
    df_photos.loc[:, "abs_path"] = os.path.abspath(f"{directorio}/{df_photos['path']}")


In [20]:
def get_photos_new_species(df_new_species, df_photos):

    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    download_photos(df_photos_new_species)
    
    df_photos_new_species.to_csv('data/new_species_photos.csv', index=False)

    return df_photos_new_species

In [21]:
get_photos_new_species(df_new_species, df_photos)

,id,photos_id,iconic_taxon,taxon_name,photos_medium_url,user_login,latitude,longitude,license_photo,attribution,path,abs_path
14,460659,621376,insecta,Euchloe crameri,https://minka-sdg.org/attachments/local_photos...,amb_platges,41.418238,2.232330,cc-by,"(c) AMB Platges, some rights reserved (CC BY)",460659_621376.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
22,460191,620801,insecta,Philanthus triangulum,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265027,1.982810,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460191_620801.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
28,460186,620797,plantae,Metrosideros excelsa,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265027,1.982810,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460186_620797.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
29,460183,620793,insecta,Plutella xylostella,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265005,1.985577,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460183_620793.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
31,460174,620778,insecta,Euodynerus variegatus,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265011,1.986323,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460174_620778.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
36,460170,620773,insecta,Beosus maritimus,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265011,1.986323,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460170_620773.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
37,460169,620772,mollusca,Otala punctata,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265011,1.986323,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460169_620772.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
52,460147,620742,plantae,Elaeagnus angustifolia,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264961,1.988037,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460147_620742.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
55,460145,620738,arachnida,Eatoniana plumipes,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264961,1.988037,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460145_620738.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...
68,460130,620722,insecta,Dicranocephalus albipes,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265015,1.990760,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",460130_620722.jpg,c:\Users\alexl\.conda\Minka_Analysis\minka_pho...


In [24]:
def get_new_species_photo_snapshot(df_obs, df_photos, last_days=30):

    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")
    df_species = df_obs[df_obs["taxon_rank"] == "species"]
    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    df_new_species = df_first[df_first["observed_on"] > cutoff_date].reset_index(drop=True)

    # Esta parte la he hecho con ChatGPT porque me daba un  error que no estaba entendiendo y a añadido la funcion map
    # lo que hace es agregar el url a la foto correspondiente basandose en el id
    df_photos_unique = df_photos.drop_duplicates(subset="id")
    photo_map = df_photos_unique.set_index("id")["photos_medium_url"]
    df_new_species["photos_medium_url"] = df_new_species["id"].map(photo_map)

    # Mapear attribution (mismo método)
    attribution_map = df_photos_unique.set_index("id")["attribution"]
    df_new_species["attribution"] = df_new_species["id"].map(attribution_map)

    df_new_species['obs_url'] = df_new_species['id'].apply(lambda x: f"https://minka-sdg.org/observations/{x}") 

    os.makedirs("data", exist_ok=True)
    
    df_new_species[["taxon_name", "observed_on", "user_login", "photos_medium_url", "attribution", "obs_url"]].to_csv("data/new_species_photo.csv", index=False)

    return df_new_species[["taxon_name", "observed_on", "user_login", "photos_medium_url", "attribution", "obs_url"]].reset_index(drop=True)

In [25]:
get_new_species_photo_snapshot(df_obs, df_photos)

,taxon_name,observed_on,user_login,photos_medium_url,attribution,obs_url
0,Streptopelia turtur,2025-05-13,mediambient_ajelprat,https://minka-sdg.org/attachments/local_photos...,"(c) mediambient_ajelprat, some rights reserved...",https://minka-sdg.org/observations/455105
1,Matthiola incana,2025-05-13,amb_platges,https://minka-sdg.org/attachments/local_photos...,"(c) AMB Platges, some rights reserved (CC BY)",https://minka-sdg.org/observations/454457
2,Petrosedum sediforme,2025-05-15,amb_platges,https://minka-sdg.org/attachments/local_photos...,"(c) AMB Platges, some rights reserved (CC BY)",https://minka-sdg.org/observations/454767
3,Eatoniana plumipes,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460145
4,Elaeagnus angustifolia,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460147
5,Otala punctata,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460169
6,Beosus maritimus,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460170
7,Euchloe crameri,2025-05-26,amb_platges,https://minka-sdg.org/attachments/local_photos...,"(c) AMB Platges, some rights reserved (CC BY)",https://minka-sdg.org/observations/460659
8,Euodynerus variegatus,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460174
9,Plutella xylostella,2025-05-26,xasalva,https://minka-sdg.org/attachments/local_photos...,"(c) xavi salvador costa, some rights reserved ...",https://minka-sdg.org/observations/460183


Con esto estaríamos creando las funciones para extraer los datos. Luego estarían las de crear los gráficos y montar el informe.